# 15 — In-game stats features (api-football)

Test rolling shots-on-target diff (and related stats: total shots, possession, pass accuracy) against the equal-weight MLP+XGB+CB+RDC ensemble baseline.

**Coverage:** api-football per-match stats start late 2018 — useful WC backtest is WC 2022 only (WC 2018 is right at the boundary, most pre-2018 training rows have no SoT).

Approach: fill missing SoT-diff with 0 (neutral) so the training set stays full. WC 2010/2014/2018 are basically running baseline; WC 2022 shows whether SoT adds signal where it exists.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
import xgboost as xgb
from catboost import CatBoostClassifier
from src.elo import EloModel
from src.draw_model import match_outcome
from src.regression_dc import RegressionDixonColesModel
from src.stats_features import attach_stats_features_to_training

## Build base features (same as notebook 12) + attach SoT-rolling features

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv  = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])
elo = EloModel()
elo_enriched = elo.fit(df_all)
df = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                  on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])

HOME_ADVANTAGE=100; FLOOR=1e5
df['neutral'] = df['neutral'].fillna(False)
df['eff_elo_diff']  = (df['home_elo_pre'] + (~df['neutral']).astype(int)*HOME_ADVANTAGE) - df['away_elo_pre']
df['elo_diff_norm'] = df['eff_elo_diff']/400.0
df['log_value_diff']    = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR)/df['away_top_n_value_eur'].clip(lower=FLOOR))
df['outfield_age_diff'] = df['home_outfield_mean_age'] - df['away_outfield_mean_age']
df['top1_share_diff']   = df['home_top1_share'] - df['away_top1_share']
df['fifa_points_diff']  = (df['home_fifa_points'].fillna(0) - df['away_fifa_points'].fillna(0))/1000.0
df['home_elo']  = df['home_elo_pre']/100.0;  df['away_elo']  = df['away_elo_pre']/100.0
df['home_logv'] = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR))
df['away_logv'] = np.log10(df['away_top_n_value_eur'].clip(lower=FLOOR))
df['home_fpts'] = df['home_fifa_points'].fillna(0)/1000.0
df['away_fpts'] = df['away_fifa_points'].fillna(0)/1000.0
df['outcome']   = df.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)

# Attach rolling SoT (and other in-game stats) over last 10 matches per team
df = attach_stats_features_to_training(df, rolling_window=10, min_periods=3,
    stats_path='../data/processed/api_lineups_stats.csv',
    matches_path='../data/processed/api_lineups_matches.csv',
    join_path='../data/processed/training_to_api_fixture.csv')
print('shape after stats join:', df.shape)
for c in ['sot_net_diff_r10','diff_sot_for_r10','diff_sot_against_r10','diff_total_shots_for_r10','diff_possession_for_r10','diff_pass_pct_for_r10']:
    nn = df[c].notna().sum()
    print(f'  {c}: {nn} non-null ({nn/len(df)*100:.1f}%)')

shape after stats join: (13067, 72)
  sot_net_diff_r10: 2544 non-null (19.5%)
  diff_sot_for_r10: 2544 non-null (19.5%)
  diff_sot_against_r10: 2544 non-null (19.5%)
  diff_total_shots_for_r10: 2545 non-null (19.5%)
  diff_possession_for_r10: 2544 non-null (19.5%)
  diff_pass_pct_for_r10: 2339 non-null (17.9%)


## Feature sets and walk-forward backtest (NaN → 0)

Fill missing stat-diffs with 0 (neutral). Pre-2018 training rows treat SoT-diff as zero-info; XGB/MLP/CB handle this fine via splits/non-linearity. Compare:
- **BASE**: 5 features (notebook 12 baseline)
- **+SoT-net**: BASE + sot_net_diff_r10 (single signal: rolling shots-on-target net diff)
- **+SoT-stack**: BASE + sot_net_diff + diff_total_shots_for + diff_pass_pct_for

In [3]:
FEAT_CLF  = ['elo_diff_norm','log_value_diff','outfield_age_diff','top1_share_diff','fifa_points_diff']
RDC_FEAT  = ['elo','logv','fpts']
FEAT_SOT  = FEAT_CLF + ['sot_net_diff_r10']
FEAT_STACK= FEAT_CLF + ['sot_net_diff_r10','diff_total_shots_for_r10','diff_pass_pct_for_r10']

needed = FEAT_CLF + [f'{s}_{c}' for s in ('home','away') for c in RDC_FEAT] + ['outcome']
valid = df.dropna(subset=needed)
valid = valid[(valid['home_top_n_value_eur']>FLOOR)&(valid['away_top_n_value_eur']>FLOOR)].copy()
for c in [c for c in df.columns if c.startswith(('sot_net_diff','diff_'))]:
    valid[c] = valid[c].fillna(0)
print(f'Valid matches: {len(valid):,}')
print(f'  with non-zero SoT-diff: {(valid["sot_net_diff_r10"]!=0).sum()} ({(valid["sot_net_diff_r10"]!=0).mean()*100:.1f}%)')
# WC matches with non-zero SoT-diff?
wc = valid[valid['tournament']=='FIFA World Cup']
for y in [2010,2014,2018,2022]:
    sub = wc[wc['date'].dt.year==y]
    nn = (sub['sot_net_diff_r10']!=0).sum()
    print(f'  WC {y}: {nn}/{len(sub)} matches with SoT data')

Valid matches: 7,641
  with non-zero SoT-diff: 2314 (30.3%)
  WC 2010: 0/64 matches with SoT data
  WC 2014: 0/64 matches with SoT data
  WC 2018: 49/64 matches with SoT data
  WC 2022: 60/64 matches with SoT data


In [4]:
def score(y_true, p):
    p = np.clip(p, 1e-6, 1-1e-6)
    oh = np.zeros_like(p); oh[np.arange(len(y_true)), y_true] = 1
    return {'log_loss': log_loss(y_true, p, labels=[0,1,2]),
            'accuracy': float((p.argmax(axis=1)==y_true).mean()),
            'brier':    float(np.mean(np.sum((p-oh)**2, axis=1)))}

def fit_xgb(X, y):
    return xgb.XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=300,
                              max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                              reg_lambda=1.0, eval_metric='mlogloss', n_jobs=4, verbosity=0).fit(X, y)
def fit_mlp(Xs, y):
    return MLPClassifier(hidden_layer_sizes=(32,16), activation='relu', max_iter=500,
                          learning_rate_init=0.005, alpha=1e-3, early_stopping=True, random_state=0).fit(Xs, y)
def fit_cb(X, y):
    return CatBoostClassifier(iterations=400, depth=4, learning_rate=0.05,
                                loss_function='MultiClass', verbose=False,
                                random_seed=0, thread_count=4).fit(X, y)
def proba_ord(clf, X):
    p = clf.predict_proba(X); classes = np.asarray(clf.classes_).flatten()
    order = [list(classes).index(c) for c in (0,1,2)]
    return p[:, order]

In [5]:
def run_backtest(feat_cols, label, with_rdc=True):
    preds = {}
    for year in [2010, 2014, 2018, 2022]:
        train = valid[valid['date'].dt.year < year]
        test  = valid[(valid['date'].dt.year == year) & (valid['tournament']=='FIFA World Cup')]
        if test.empty: continue
        y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
        Xtr = train[feat_cols].to_numpy(); Xte = test[feat_cols].to_numpy()
        sc = StandardScaler().fit(Xtr)
        p_mlp = proba_ord(fit_mlp(sc.transform(Xtr), y_train), sc.transform(Xte))
        p_xgb = proba_ord(fit_xgb(Xtr, y_train), Xte)
        p_cb  = proba_ord(fit_cb(Xtr, y_train), Xte)
        rdc_preds = None
        if with_rdc:
            rdc = RegressionDixonColesModel(RDC_FEAT, xi=0.00038).fit(train)
            rdc_preds = rdc.predict_many(test)[['p_away_win','p_draw','p_home_win']].to_numpy()
        preds[year] = {'y': y_test, 'mlp': p_mlp, 'xgb': p_xgb, 'cb': p_cb, 'rdc': rdc_preds}
    # Per-WC scores for the ensemble
    rows = []
    models = ['mlp','xgb','cb'] + (['rdc'] if with_rdc else [])
    for y, d in preds.items():
        avg = np.mean([d[m] for m in models], axis=0); avg /= avg.sum(axis=1, keepdims=True)
        s = score(d['y'], avg)
        rows.append({'wc': y, 'n': len(d['y']), **s})
    # Aggregate
    yall = np.concatenate([d['y'] for d in preds.values()])
    pall = np.vstack([(np.mean([d[m] for m in models], axis=0) / np.mean([d[m] for m in models], axis=0).sum(axis=1, keepdims=True)) for d in preds.values()])
    s_all = score(yall, pall)
    print(f'\n== {label} (feat: {len(feat_cols)} cols, RDC={"yes" if with_rdc else "no"}) ==')
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print(f'  Aggregate: LL={s_all["log_loss"]:.4f}  acc={s_all["accuracy"]:.4f}  brier={s_all["brier"]:.4f}')
    return preds, s_all

preds_base, s_base = run_backtest(FEAT_CLF, 'BASELINE')
preds_sot,  s_sot  = run_backtest(FEAT_SOT, '+SoT-net')
preds_stk,  s_stk  = run_backtest(FEAT_STACK, '+SoT-stack')


== BASELINE (feat: 5 cols, RDC=yes) ==
  wc  n  log_loss  accuracy  brier
2010 64    0.9762    0.4688 0.5844
2014 64    0.9576    0.5625 0.5700
2018 64    0.9801    0.5625 0.5898
2022 64    1.0588    0.5469 0.6165
  Aggregate: LL=0.9932  acc=0.5352  brier=0.5902



== +SoT-net (feat: 6 cols, RDC=yes) ==
  wc  n  log_loss  accuracy  brier
2010 64    0.9800    0.4688 0.5866
2014 64    0.9526    0.5781 0.5658
2018 64    0.9952    0.5156 0.5955
2022 64    1.0601    0.5156 0.6124
  Aggregate: LL=0.9970  acc=0.5195  brier=0.5901



== +SoT-stack (feat: 8 cols, RDC=yes) ==
  wc  n  log_loss  accuracy  brier
2010 64    0.9777    0.4688 0.5848
2014 64    0.9539    0.5469 0.5667
2018 64    1.0055    0.5312 0.6037
2022 64    1.0404    0.5469 0.6014
  Aggregate: LL=0.9944  acc=0.5234  brier=0.5892


## WC 2022 isolation

Most of the SoT signal lives in WC 2022 (3+ years of prior api-football data). Compare just that year.

In [6]:
def wc22_summary(preds, label):
    d = preds[2022]
    models = [m for m in ('mlp','xgb','cb','rdc') if d.get(m) is not None]
    avg = np.mean([d[m] for m in models], axis=0); avg /= avg.sum(axis=1, keepdims=True)
    s = score(d['y'], avg)
    return {'feat': label, 'log_loss': s['log_loss'], 'accuracy': s['accuracy'], 'brier': s['brier']}

rows = [wc22_summary(preds_base, 'BASELINE'),
        wc22_summary(preds_sot,  '+SoT-net'),
        wc22_summary(preds_stk,  '+SoT-stack')]
print('WC 2022 ensemble (MLP+XGB+CB+RDC) hold-out:')
print(pd.DataFrame(rows).round(4).to_string(index=False))

WC 2022 ensemble (MLP+XGB+CB+RDC) hold-out:
      feat  log_loss  accuracy  brier
  BASELINE    1.0588    0.5469 0.6165
  +SoT-net    1.0601    0.5156 0.6124
+SoT-stack    1.0404    0.5469 0.6014


## XGB feature importance

In [7]:
last_train = valid[valid['date'].dt.year < 2022]
for cols, label in [(FEAT_CLF, 'BASELINE'), (FEAT_SOT, '+SoT-net'), (FEAT_STACK, '+SoT-stack')]:
    clf = fit_xgb(last_train[cols].to_numpy(), last_train['outcome'].to_numpy())
    imp = pd.Series(clf.feature_importances_, index=cols).sort_values(ascending=False)
    print(f'\n{label}:')
    print(imp.round(3).to_string())


BASELINE:
elo_diff_norm        0.352
fifa_points_diff     0.237
log_value_diff       0.191
top1_share_diff      0.113
outfield_age_diff    0.108



+SoT-net:
elo_diff_norm        0.322
fifa_points_diff     0.217
log_value_diff       0.172
sot_net_diff_r10     0.099
top1_share_diff      0.096
outfield_age_diff    0.094



+SoT-stack:
elo_diff_norm               0.288
fifa_points_diff            0.178
log_value_diff              0.140
diff_total_shots_for_r10    0.081
top1_share_diff             0.080
sot_net_diff_r10            0.079
outfield_age_diff           0.077
diff_pass_pct_for_r10       0.076
